# UMich / Deep Blue Fast-Formation Extract Notebook

This notebook is the Colab-side audit and feature extraction step for the Weng/Mohtat/Stefanopoulou fast-formation dataset.

Expected Drive layout:

`/content/drive/MyDrive/Michigan_DeepBlue/fast-formation/`

The notebook is intentionally audit-first. It recursively scans the nested `data/` folders, prioritizes `2020-10-aging-test-cycles`, detects cycle and discharge-capacity columns, verifies that at least 30 cells have `Q0` and `cycle_life` at `0.85 * Q0`, then writes UMich feature CSVs compatible with the existing four-dataset pipeline.

## 0. Configuration

In [ ]:
# Edit only if your Drive folder or branch name differs.
GITHUB_REPO = "https://github.com/osmansafacifci/Graduation-Project-Dicle.git"
GITHUB_BRANCH = "main"
REPO_DIR = "/content/Graduation-Project-Dicle"

UMICH_ROOT = "/content/drive/MyDrive/Michigan_DeepBlue/fast-formation"
PREFERRED_DATA_SUBFOLDER = "2020-10-aging-test-cycles"

DATASET_LABEL = "umich"
N_WINDOWS = (50, 100)
EOL_FRACTION = 0.85
MIN_MODELED_CELLS = 30

# Capacity sanity gates for NMC/graphite pouch cells. Values outside this range
# are not automatically discarded, but they are flagged in the audit.
EXPECTED_CAPACITY_AH_MIN = 0.5
EXPECTED_CAPACITY_AH_MAX = 8.0


## 1. Mount Drive, clone repo, install lightweight dependencies

In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f"[drive] not running in Colab or Drive already mounted: {exc}")

REPO_DIR = Path(REPO_DIR)
if REPO_DIR.exists():
    print(f"[repo] using existing {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "pandas", "numpy", "tqdm"], check=True)

PROJECT_ROOT = REPO_DIR
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

UMICH_ROOT = Path(UMICH_ROOT)
DATA_ROOT = UMICH_ROOT / "data"
print("[root]", UMICH_ROOT)
print("[data]", DATA_ROOT)
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Could not find UMich data folder: {DATA_ROOT}")


## 2. Recursive folder audit

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

def rel(path: Path) -> str:
    try:
        return str(path.relative_to(UMICH_ROOT))
    except Exception:
        return str(path)

print("[tree] top-level folders")
for p in sorted(UMICH_ROOT.iterdir()):
    if p.is_dir():
        print("  DIR ", rel(p))
    else:
        print("  FILE", rel(p))

print("\n[tree] data subfolders")
for p in sorted(DATA_ROOT.iterdir()):
    if p.is_dir():
        n_csv = len(list(p.rglob('*.csv')))
        n_txt = len(list(p.rglob('*.txt')))
        print(f"  {p.name:45s} csv={n_csv:5d} txt={n_txt:5d}")

csv_files = sorted(DATA_ROOT.rglob("*.csv"))
print(f"\n[files] found {len(csv_files)} CSV files under data/")
display(pd.DataFrame({"path": [rel(p) for p in csv_files[:80]], "size_mb": [p.stat().st_size / 1e6 for p in csv_files[:80]]}))


## 3. Detect cycle-summary files and columns

The preferred source is `2020-10-aging-test-cycles`, because it should contain one row per aging cycle. If that folder is not usable, the detector falls back to all CSV files and can also aggregate time-series rows by cycle using the maximum discharge capacity within each cycle.

In [ ]:
from collections import Counter

def normalize_name(name) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")

def safe_read_csv(path: Path, nrows=None):
    last_exc = None
    for enc in ("utf-8", "utf-8-sig", "latin1"):
        try:
            return pd.read_csv(path, nrows=nrows, low_memory=False, encoding=enc)
        except Exception as exc:
            last_exc = exc
    raise last_exc

def choose_cycle_column(columns):
    norms = {c: normalize_name(c) for c in columns}
    preferred = [
        "cycle_index", "cycle_number", "cycle_num", "cycle", "cycle_count",
        "cycle_id", "cycle_no", "cycle_index_number",
    ]
    for target in preferred:
        for c, n in norms.items():
            if n == target:
                return c
    for c, n in norms.items():
        if "cycle" in n and not any(bad in n for bad in ("life", "count_through", "time")):
            return c
    return None

def choose_discharge_capacity_column(columns):
    norms = {c: normalize_name(c) for c in columns}
    scored = []
    for c, n in norms.items():
        score = 0
        if "discharge" in n or "dchg" in n or "disch" in n:
            score += 3
        if "capacity" in n or re.search(r"(^|_)cap($|_)", n):
            score += 3
        if "charge_capacity" in n and "discharge" not in n and "dchg" not in n:
            score -= 4
        if any(unit in n for unit in ("ah", "mah")):
            score += 1
        if any(bad in n for bad in ("energy", "specific", "retention", "percent", "pct")):
            score -= 2
        if score >= 5:
            scored.append((score, c, n))
    if not scored:
        return None
    return sorted(scored, reverse=True)[0][1]

def choose_cell_column(columns):
    norms = {c: normalize_name(c) for c in columns}
    preferred = ["cell_id", "cell", "barcode", "serial_number", "sample", "test_name", "device", "channel"]
    for target in preferred:
        for c, n in norms.items():
            if n == target:
                return c
    for c, n in norms.items():
        if any(tok in n for tok in ("cell", "barcode", "sample", "channel")):
            return c
    return None

def derive_cell_id(path: Path) -> str:
    stem = normalize_name(path.stem)
    parent = normalize_name(path.parent.name)
    if parent in {"data", normalize_name(PREFERRED_DATA_SUBFOLDER)}:
        return stem
    return f"{parent}_{stem}"

def inspect_file(path: Path):
    out = {
        "path": rel(path),
        "size_mb": path.stat().st_size / 1e6,
        "preferred_folder": PREFERRED_DATA_SUBFOLDER in str(path),
        "read_ok": False,
        "n_columns": 0,
        "cycle_col": None,
        "qdis_col": None,
        "cell_col": None,
        "score": 0,
        "error": "",
    }
    try:
        sample = safe_read_csv(path, nrows=300)
        out["read_ok"] = True
        out["n_columns"] = len(sample.columns)
        out["cycle_col"] = choose_cycle_column(sample.columns)
        out["qdis_col"] = choose_discharge_capacity_column(sample.columns)
        out["cell_col"] = choose_cell_column(sample.columns)
        out["score"] = int(out["cycle_col"] is not None) * 4 + int(out["qdis_col"] is not None) * 6 + int(out["preferred_folder"]) * 3
    except Exception as exc:
        out["error"] = repr(exc)[:250]
    return out

inspect_rows = [inspect_file(p) for p in csv_files]
inspect_df = pd.DataFrame(inspect_rows).sort_values(["score", "preferred_folder", "size_mb"], ascending=[False, False, False])
display(inspect_df.head(40))

usable = inspect_df[(inspect_df["cycle_col"].notna()) & (inspect_df["qdis_col"].notna())].copy()
preferred = usable[usable["preferred_folder"]].copy()
selected_df = preferred if len(preferred) else usable
print(f"[detect] usable CSVs={len(usable)}, selected={len(selected_df)}")
if selected_df.empty:
    print("No usable cycle/capacity CSV detected. Showing unique column sets from readable files:")
    display(inspect_df[inspect_df['read_ok']].head(30))
    raise RuntimeError("No cycle/discharge-capacity CSVs detected. Inspect the columns above and adjust the detector.")


## 4. Build tidy per-cycle discharge-capacity table

In [ ]:
def clean_cell_id(value) -> str:
    text = normalize_name(value)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown_cell"

def capacity_to_ah(values: pd.Series):
    vals = pd.to_numeric(values, errors="coerce")
    median = float(vals[(vals > 0) & np.isfinite(vals)].median()) if vals.notna().any() else np.nan
    unit = "Ah"
    if np.isfinite(median) and median > 50:
        vals = vals / 1000.0
        unit = "mAh_to_Ah"
    return vals, unit

series_records = []
tidy_rows = []
metadata_rows = []

path_by_rel = {rel(p): p for p in csv_files}
for _, info in selected_df.iterrows():
    path = path_by_rel[info["path"]]
    cycle_col = info["cycle_col"]
    qdis_col = info["qdis_col"]
    cell_col = info["cell_col"]
    try:
        df = safe_read_csv(path)
    except Exception as exc:
        metadata_rows.append({"source_file": rel(path), "parse_status": "error:read", "error_message": repr(exc)})
        continue

    if cycle_col not in df.columns or qdis_col not in df.columns:
        metadata_rows.append({"source_file": rel(path), "parse_status": "error:missing_detected_columns", "error_message": ""})
        continue

    df = df.copy()
    df["__cycle"] = pd.to_numeric(df[cycle_col], errors="coerce")
    df["__qdis"], unit = capacity_to_ah(df[qdis_col])
    if cell_col is not None and cell_col in df.columns:
        df["__cell_id"] = df[cell_col].map(clean_cell_id)
    else:
        df["__cell_id"] = clean_cell_id(derive_cell_id(path))

    df = df[np.isfinite(df["__cycle"]) & np.isfinite(df["__qdis"]) & (df["__qdis"] > 0)].copy()
    if df.empty:
        metadata_rows.append({"source_file": rel(path), "parse_status": "error:no_positive_capacity_rows", "error_message": ""})
        continue

    for cell_id, g in df.groupby("__cell_id"):
        per_cycle = g.groupby("__cycle", as_index=False)["__qdis"].max().sort_values("__cycle")
        per_cycle = per_cycle.drop_duplicates("__cycle")
        qd = per_cycle["__qdis"].to_numpy(dtype=float)
        raw_cycles = per_cycle["__cycle"].to_numpy(dtype=float)
        if len(qd) < 5:
            continue
        source_score = int(info["preferred_folder"]) * 100000 + len(qd)
        series_records.append({
            "cell_id": cell_id,
            "source_file": rel(path),
            "source_score": source_score,
            "n_points": len(qd),
            "qdis": qd,
            "raw_cycles": raw_cycles,
            "capacity_unit_action": unit,
        })

        metadata_rows.append({
            "source_file": rel(path),
            "cell_id": cell_id,
            "parse_status": "ok",
            "cycle_col": cycle_col,
            "qdis_col": qdis_col,
            "cell_col": cell_col or "derived_from_filename",
            "capacity_unit_action": unit,
            "n_points": len(qd),
            "raw_cycle_min": float(np.nanmin(raw_cycles)),
            "raw_cycle_max": float(np.nanmax(raw_cycles)),
            "qdis_median_ah": float(np.nanmedian(qd)),
        })

if not series_records:
    metadata = pd.DataFrame(metadata_rows)
    display(metadata.head(50))
    raise RuntimeError("No UMich cell traces parsed successfully. Inspect metadata above.")

# If the same cell appears in multiple folders/files, keep the preferred-folder / longest trace.
best = {}
for rec in series_records:
    current = best.get(rec["cell_id"])
    if current is None or rec["source_score"] > current["source_score"]:
        best[rec["cell_id"]] = rec

qd_by_cell = {cell_id: rec["qdis"] for cell_id, rec in sorted(best.items())}

for cell_id, rec in sorted(best.items()):
    for idx, (raw_cycle, q) in enumerate(zip(rec["raw_cycles"], rec["qdis"]), start=1):
        tidy_rows.append({
            "dataset": DATASET_LABEL,
            "cell_id": cell_id,
            "cycle_index": idx,
            "raw_cycle_index": raw_cycle,
            "Q_discharge": q,
            "source_file": rec["source_file"],
        })

tidy = pd.DataFrame(tidy_rows)
metadata = pd.DataFrame(metadata_rows)

tidy_path = INTERMEDIATE_DIR / "umich_cycles_tidy.csv"
metadata_path = INTERMEDIATE_DIR / "umich_cell_metadata.csv"
tidy.to_csv(tidy_path, index=False)
metadata.to_csv(metadata_path, index=False)

print(f"[parsed cells] {len(qd_by_cell)}")
print(f"[save] {tidy_path}")
print(f"[save] {metadata_path}")
display(tidy.head())
display(metadata[metadata['parse_status'].eq('ok')].head(20))


## 5. EOL audit and kill gate

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "1_features"))
from build_features import compute_q0, compute_cycle_life, build_feature_rows

audit_rows = []
for cell_id, qd in sorted(qd_by_cell.items()):
    q0 = compute_q0(qd)
    cycle_life = compute_cycle_life(qd, q0, EOL_FRACTION)
    q_median = float(np.nanmedian(qd)) if len(qd) else np.nan
    audit_rows.append({
        "dataset": DATASET_LABEL,
        "cell_id": cell_id,
        "n_cycles_observed": len(qd),
        "q0": q0,
        "cycle_life_0p85": cycle_life,
        "cycle_life": cycle_life,
        "is_censored": int(not np.isfinite(cycle_life)),
        "capacity_median_ah": q_median,
        "capacity_sanity_flag": int(not (EXPECTED_CAPACITY_AH_MIN <= q_median <= EXPECTED_CAPACITY_AH_MAX)),
        "eligible_N100": int(len(qd) >= 100 and np.isfinite(q0) and q0 > 0 and np.isfinite(cycle_life)),
    })

audit = pd.DataFrame(audit_rows).sort_values("cell_id")
audit_path = INTERMEDIATE_DIR / "umich_cell_audit.csv"
audit.to_csv(audit_path, index=False)

summary = audit.agg(
    n_cells=("cell_id", "count"),
    n_eligible_N100=("eligible_N100", "sum"),
    n_censored=("is_censored", "sum"),
    n_capacity_sanity_flags=("capacity_sanity_flag", "sum"),
).reset_index(drop=True)
threshold_path = INTERMEDIATE_DIR / "umich_threshold_summary.csv"
summary.to_csv(threshold_path, index=False)

print(f"[save] {audit_path}")
print(f"[save] {threshold_path}")
display(summary)
display(audit.head(50))

n_eligible = int(audit["eligible_N100"].sum())
if n_eligible < MIN_MODELED_CELLS:
    raise RuntimeError(
        f"KILL GATE FAILED: only {n_eligible} cells have N>=100, valid Q0, and observed 0.85*Q0 cycle_life. "
        f"Need at least {MIN_MODELED_CELLS}. Do not use UMich as S1 without revising the contract."
    )
print(f"[kill gate passed] {n_eligible} modeled cells available for N=100")


## 6. Build raw and Q0-normalized 34-feature tables

In [ ]:
raw_rows = build_feature_rows(
    qd_by_cell,
    dataset=DATASET_LABEL,
    n_windows=tuple(N_WINDOWS),
    eol_fraction=EOL_FRACTION,
    capacity_normalize=False,
)
capnorm_rows = build_feature_rows(
    qd_by_cell,
    dataset=DATASET_LABEL,
    n_windows=tuple(N_WINDOWS),
    eol_fraction=EOL_FRACTION,
    capacity_normalize=True,
)

features = pd.DataFrame(raw_rows)
features_capnorm = pd.DataFrame(capnorm_rows)

features_path = INTERMEDIATE_DIR / "features_sop12_umich.csv"
features_capnorm_path = INTERMEDIATE_DIR / "features_sop12_umich_capnorm.csv"
features.to_csv(features_path, index=False)
features_capnorm.to_csv(features_capnorm_path, index=False)

print(f"[save] {features_path} rows={len(features)}")
print(f"[save] {features_capnorm_path} rows={len(features_capnorm)}")
display(features.groupby(["n_cycles", "capacity_normalized"])[["cell_id", "is_censored"]].agg(cell_rows=("cell_id", "count"), censored=("is_censored", "sum")).reset_index())
display(features.head())


## 7. Optional combined tables for local S1 validation

In [ ]:
four_raw_path = INTERMEDIATE_DIR / "features_sop12_four_dataset.csv"
four_capnorm_path = INTERMEDIATE_DIR / "features_sop12_four_dataset_capnorm.csv"

if four_raw_path.exists():
    four_raw = pd.read_csv(four_raw_path)
    combined_plus = pd.concat([four_raw, features[four_raw.columns]], ignore_index=True)
    out = INTERMEDIATE_DIR / "features_sop12_four_dataset_plus_umich.csv"
    combined_plus.to_csv(out, index=False)
    print(f"[save] {out} rows={len(combined_plus)}")
else:
    print(f"[skip] missing {four_raw_path}")

if four_capnorm_path.exists():
    four_capnorm = pd.read_csv(four_capnorm_path)
    combined_plus_capnorm = pd.concat([four_capnorm, features_capnorm[four_capnorm.columns]], ignore_index=True)
    out = INTERMEDIATE_DIR / "features_sop12_four_dataset_plus_umich_capnorm.csv"
    combined_plus_capnorm.to_csv(out, index=False)
    print(f"[save] {out} rows={len(combined_plus_capnorm)}")
else:
    print(f"[skip] missing {four_capnorm_path}")


## 8. Package outputs for download

In [ ]:
from datetime import datetime
import zipfile

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
zip_path = Path(f"/content/umich_extract_outputs_{timestamp}.zip")
output_names = [
    "umich_cycles_tidy.csv",
    "umich_cell_audit.csv",
    "umich_cell_metadata.csv",
    "umich_threshold_summary.csv",
    "features_sop12_umich.csv",
    "features_sop12_umich_capnorm.csv",
    "features_sop12_four_dataset_plus_umich.csv",
    "features_sop12_four_dataset_plus_umich_capnorm.csv",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in output_names:
        p = INTERMEDIATE_DIR / name
        if p.exists():
            zf.write(p, arcname=f"data/intermediate/{name}")

print(f"[zip] {zip_path}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print(f"Manual download from Colab Files panel: {zip_path} ({exc})")
